# Open-answer arithmetic on Kaggle: can the models calculate without options?

The same 144 expressions as the easy multiple-choice test (`+ - * / **`, `a + b * c`), but
**no options**: the model writes the number itself, and it is checked against the exact value.

Two conditions per model:
- **number**: "Answer with the number only." One step, like the letter score.
- **working**: "Show your working briefly, then `Answer: <number>`." Room to compute.

Qwen2.5-3B-Instruct and Phi-4-mini-instruct, greedy decoding, no system message.

**Before running** (right-hand panel, *Session options*):
- *Accelerator*: **GPU T4 x2** (one GPU is used) or GPU P100
- *Internet*: **on**
- *Input*: add the dataset made from `kaggle_easy_open_files.zip`

Results: `/kaggle/working/Thesis/results/easy_open/`, zipped at the end as
`/kaggle/working/easy_open_results.zip` (*Output* panel). About 10 min.

In [ ]:
import os, sys, glob, shutil, subprocess
INPUT = os.environ.get('KAGGLE_INPUT_DIR', '/kaggle/input')        # read-only dataset mount
WORK = os.environ.get('KAGGLE_WORK_DIR', '/kaggle/working/Thesis')  # writable, kept as output
found = glob.glob(os.path.join(INPUT, '**', 'open_arithmetic.py'), recursive=True)
assert found, f'project files not found under {INPUT}: add the dataset as an Input of this notebook'
SRC = os.path.dirname(found[0])
shutil.copytree(SRC, WORK, dirs_exist_ok=True)
os.chdir(WORK)
for f in ['open_arithmetic.py', 'baseline_letter_logprobs.py', 'easy_open/easy_open_test.jsonl', 'ivp_mcq/grade_mcq.py']:
    assert os.path.exists(f), f'missing {f} in the uploaded dataset (keep the folder structure of the zip)'
print('project files copied from', SRC, 'to', WORK)
if INPUT == '/kaggle/input':
    r = subprocess.run('curl -s -o /dev/null -w "%{http_code}" https://huggingface.co', shell=True,
                       capture_output=True, text=True)
    assert r.stdout.strip() in ('200', '301', '302'), 'no internet: turn Internet on in Session options'
    subprocess.run('pip -q install "transformers>=4.51" accelerate', shell=True)
import torch, transformers
if INPUT == '/kaggle/input':
    assert torch.cuda.is_available(), 'no GPU: set Accelerator to GPU T4 x2 or P100 in Session options'
print('cwd', os.getcwd(), '| torch', torch.__version__, '| transformers', transformers.__version__,
      '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
MODELS = ['qwen2.5-3b-instruct', 'phi-4-mini-instruct']
MODES = ['number', 'working']
LIMIT = None          # leave None for the real run
OUT = 'results/easy_open'

## Run both conditions for both models

In [ ]:
dirs = []
for m in MODELS:
    for mode in MODES:
        cmd = f'{sys.executable} open_arithmetic.py --model {m} --mode {mode} --outdir {OUT}' + (f' --limit {LIMIT}' if LIMIT else '')
        print('>>', cmd, flush=True)
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        line = [l for l in r.stdout.splitlines() if 'acc=' in l]
        if r.returncode or not line:
            raise RuntimeError('run failed:\n' + '\n'.join(r.stderr.strip().splitlines()[-10:]))
        print('  ', line[-1].split('  -> ')[0], flush=True)
        dirs.append(line[-1].split('  -> ')[1].strip())

## Accuracy per operation

In [ ]:
subprocess.run(f'{sys.executable} open_arithmetic.py --table ' + ' '.join(f'"{d}/summary.json"' for d in dirs), shell=True)
import json
for d in dirs:
    s = json.load(open(f'{d}/summary.json'))
    print(f"\n--- {s['meta']['model_key']} [{s['meta']['mode']}]: first wrong answers")
    for w in s['wrong_examples'][:6]:
        print(f"  {w['expression']:12s} = {w['value']:<6} read {w['read']}  | reply: {w['reply'][:90]!r}")

## Download the results

In [ ]:
zip_base = os.path.join(os.path.dirname(WORK), 'easy_open_results')
path = shutil.make_archive(zip_base, 'zip', root_dir=WORK, base_dir=OUT)
print('wrote', path, f'({os.path.getsize(path) / 1e6:.2f} MB) - download it from the Output panel on the right')